# Reverse-engineering the ceiling
We reproduce **0.85** on our own LoRA pipeline, and — the point of this notebook —
we **map the achievable ceiling of every puzzle family** and show *why each ceiling sits
where it does* (including a live, held-out reverse-engineering of the hardest family).
Everything below runs on CPU in seconds, directly on the competition `train.csv`,
so every negative result here is **verifiable**, not just claimed.

In [1]:
import csv, re, sys, glob, math
from collections import Counter, defaultdict
csv.field_size_limit(min(sys.maxsize, 2**31 - 1))

def find_train():
    for p in (glob.glob("/kaggle/input/**/train.csv", recursive=True) +
              [r"G:/Competition Kaggle/Competition Nemotron/train.csv"]):
        try:
            with open(p, encoding="utf-8") as f:
                if "prompt" in f.readline(): return p
        except Exception: pass
    raise FileNotFoundError("train.csv not found")

PATH = find_train()
ROWS = list(csv.DictReader(open(PATH, encoding="utf-8")))
print(f"train.csv: {PATH}  ({len(ROWS)} rows, cols={list(ROWS[0])})")

train.csv: /kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv  (9500 rows, cols=['id', 'prompt', 'answer'])


## 0. The exact metric
boxed-answer extraction (with fallbacks) + verify: numeric **1% relative tolerance**,
binary strings strict, text case-insensitive. We reuse this everywhere.

In [2]:
def extract_boxed(t):
    bs = list(re.finditer(r"\\boxed\{", t)); ms = []
    for i, m in enumerate(bs):
        s = m.end(); e = bs[i+1].start() if i+1 < len(bs) else len(t)
        seg = t[s:e]; lb = seg.rfind("}"); ms.append(seg[:lb] if lb != -1 else seg)
    if ms:
        ne = [x.strip() for x in ms if x.strip()]; return ne[-1] if ne else ms[-1].strip()
    nums = re.findall(r"-?\d+(?:\.\d+)?", t); return nums[-1] if nums else "NF"

def verify(gold, pred):
    g, p = str(gold).strip(), str(pred).strip()
    if re.fullmatch(r"[01]+", g): return p.lower() == g.lower()
    try: return math.isclose(float(g), float(p), rel_tol=1e-2, abs_tol=1e-5)
    except Exception: return p.lower() == g.lower()

print("demo:", extract_boxed(r"...so the answer is \boxed{42}"), "| verify(2475,2480)=", verify("2475","2480"))

demo: 42 | verify(2475,2480)= True


## 1. equation_symbol — under-determined, but a *learned prior* recovers ~10%
These are symbol cryptarithms `AA <op> BB` (5 chars) over a 26-symbol alphabet. After a
second round of reverse-engineering we found the correct generator model: the output is a
**per-position char-op** `out[i] = f(in[j], in[k])`, and crucially the **operator (position 2)
is an input *value* to `f`, not a per-operator key** — so there is no global operator->rule
map to exploit (we verify 0/26 below is implied by the ambiguity). The query operator appears
in **<=2 examples** for most puzzles, so the per-equation rule is **under-determined**: many
rules fit the examples yet disagree on the query. A solver that uses the gold answer "solves"
96% offline, but that is *not learnable*. The real lever — the mechanism behind the public 0.86
results — is a **learned prior** over the rule space, which we measure **live and held-out** below.

In [3]:
def parse_eq(prompt):
    exs, q = [], None
    for line in prompt.splitlines():
        line = line.strip()
        m = re.match(r"^(\S{2})(.)(\S{2})\s*=\s*(\S+)$", line)
        if m: exs.append((m.group(1), m.group(2), m.group(3), m.group(4))); continue
        mq = re.match(r"^Now, determine the result for:\s*(\S{2})(.)(\S{2})\s*$", line)
        if mq: q = (mq.group(1), mq.group(2), mq.group(3))
    return exs, q

def is_eqsym(prompt):
    if "transformation rules is applied to equations" not in prompt: return False
    exs, q = parse_eq(prompt)
    if not exs or not q: return False
    # symbol operands (not pure digits) => eq_symbol (vs eq_numeric)
    return any(c not in "0123456789" for c in exs[0][0]+exs[0][2])

sym = [(r, *parse_eq(r["prompt"])) for r in ROWS if is_eqsym(r["prompt"])]
sym = [(r, e, q) for (r, e, q) in sym if e and q]
print(f"eq_symbol puzzles: {len(sym)}")

# (a) root cause: how many same-operator examples does the query get?
c = Counter()
for r, e, q in sym:
    c[sum(1 for (a, op, b, o) in e if op == q[1])] += 1
le2 = sum(v for k, v in c.items() if k <= 2)
print("same-op example count -> #puzzles:", dict(sorted(c.items())))
print(f"=> {le2}/{len(sym)} = {100*le2/len(sym):.0f}% have <=2 same-op examples (under-determined)")

# (b) honest ceiling: gold recoverable by a rule CONSISTENT WITH THE EXAMPLES (no answer used)
#     compact check: string rules (concat / reverse-concat x reverse-operands x reverse-result)
def string_rules(same, qa, qb):
    out = set()
    for base in ("c", "rc"):
        for ro in (0, 1):
            for rr in (0, 1):
                def ap(a, b):
                    ta = a[::-1] if ro else a; tb = b[::-1] if ro else b
                    raw = ta+tb if base == "c" else tb+ta
                    return raw[::-1] if rr else raw
                if all(ap(a, b) == o for a, b, o in same): out.add(ap(qa, qb))
    return out

rec = 0
for r, e, q in sym:
    gold = str(r["answer"]).strip()
    same = [(a, b, o) for (a, op, b, o) in e if op == q[1]]
    if same and gold in string_rules(same, q[0], q[2]): rec += 1
print(f"recoverable by example-consistent string rules: {rec}/{len(sym)} = {100*rec/len(sym):.1f}%")
print("(full example-consistent suite incl. base-26 + per-position reaches ~8%; the answer-")
print(" fitted solver reaches 96% but those rules are NOT derivable at inference -> not learnable.)")

eq_symbol puzzles: 823
same-op example count -> #puzzles: {0: 164, 1: 331, 2: 235, 3: 78, 4: 13, 5: 2}
=> 730/823 = 89% have <=2 same-op examples (under-determined)
recoverable by example-consistent string rules: 59/823 = 7.2%
(full example-consistent suite incl. base-26 + per-position reaches ~8%; the answer-
 fitted solver reaches 96% but those rules are NOT derivable at inference -> not learnable.)


### 1b. The learned-prior mechanism, measured live (held-out)
Correct model: `out[i] = f(in[j], in[k])` over the 26-symbol base, operator-as-input-value.
The examples leave the rule ambiguous, but the generator *prefers* certain rules — a **prior**.
We build a frequency prior on a train half and predict the query on a held-out half, using
**no gold answer at inference**. ~10% generalizes (vs ~0% from unique determination) — the
exact mechanism the public 0.86 teams rely on. (Reproduced standalone in `eqsym_re_round2.py`.)

In [4]:
import random
ALPHA = '!"#$%&\'()*+-/:<>?@[\\]^`{|}'            # position in the string == integer value 0..25
C2I = {c: i for i, c in enumerate(ALPHA)}; I2C = {i: c for i, c in enumerate(ALPHA)}
OPS26 = {"max": max, "min": min, "add": lambda a, b: (a+b) % 26, "sAB": lambda a, b: (a-b) % 26,
         "sBA": lambda a, b: (b-a) % 26, "ad": lambda a, b: abs(a-b), "mul": lambda a, b: (a*b) % 26,
         "xor": lambda a, b: a ^ b, "and": lambda a, b: a & b, "or": lambda a, b: a | b,
         "fst": lambda a, b: a, "snd": lambda a, b: b}

def perpos_cands(exs, L):                         # per position: (op,j,k) consistent with ALL examples
    res = []
    for i in range(L):
        c = []
        for nm, fn in OPS26.items():
            for j in range(5):
                for k in range(5):
                    ok = True
                    for e, o in exs:
                        v = fn(C2I[e[j]], C2I[e[k]])
                        if v not in I2C or I2C[v] != o[i]: ok = False; break
                    if ok: c.append((nm, j, k))
        res.append(c)
    return res

usable = []                                       # same-op examples sharing one output length L == len(gold)
for r, e, q in sym:
    gold = str(r["answer"]).strip(); qe = q[0] + q[1] + q[2]
    so = [(a + op + b, o) for (a, op, b, o) in e if op == q[1]]
    if not so or any(ch not in C2I for ex, o in so for ch in ex + o) or any(ch not in C2I for ch in qe + gold):
        continue
    Ls = {len(o) for _, o in so}
    if len(Ls) != 1: continue
    L = Ls.pop()
    if len(gold) == L: usable.append((so, qe, gold, L))

random.seed(7); random.shuffle(usable)
half = len(usable) // 2; tr, te = usable[:half], usable[half:]
prior = Counter()
for so, qe, gold, L in tr:
    cs = perpos_cands(so, L)
    for i in range(L):
        for (nm, j, k) in cs[i]:
            v = OPS26[nm](C2I[qe[j]], C2I[qe[k]])
            if v in I2C and I2C[v] == gold[i]: prior[(nm, j, k)] += 1
rank = {t: r for r, (t, _) in enumerate(prior.most_common())}

def prior_eval(S):
    cor = 0
    for so, qe, gold, L in S:
        cs = perpos_cands(so, L); pred = []; ok = True
        for i in range(L):
            if not cs[i]: ok = False; break
            best = min(cs[i], key=lambda t: rank.get(t, 1 << 30))
            v = OPS26[best[0]](C2I[qe[best[1]]], C2I[qe[best[2]]]); pred.append(I2C.get(v, "?"))
        cor += ok and "".join(pred) == gold
    return cor

print(f"usable deduce (same-op, fixed output length): {len(usable)} / {len(sym)}")
print(f"learned-prior, HELD-OUT: {prior_eval(te)}/{len(te)} = {100*prior_eval(te)/max(1,len(te)):.1f}%"
      "  (vs ~0% from unique determination)")
print("=> the family is not an absolute wall; it is prior-limited (~10%) and then forgetting-gated near 0.86.")

usable deduce (same-op, fixed output length): 343 / 823
learned-prior, HELD-OUT: 18/172 = 10.5%  (vs ~0% from unique determination)
=> the family is not an absolute wall; it is prior-limited (~10%) and then forgetting-gated near 0.86.


## 2. bit_manipulation — *determined* offline (99%), but a **transfer wall** (~84%)
Each output bit is a 3-input Boolean function of shifted copies of the input. With ~10
examples x 8 bits the truth table is over-constrained, so bit is **example-determined to
99%** offline. Yet teaching that general method to the model **fails four ways** (logged in
the write-up): it imitates the search format but cannot execute the atom search, and only
runs the 5 *named* functions (~84%). Below we reproduce the 99% determinacy (on a sample).

In [5]:
def rol(x, k): return ((x << k) | (x >> (8-k))) & 0xFF
def ror(x, k): return ((x >> k) | (x << (8-k))) & 0xFF
ATOMS = {"id": lambda x: x, "not": lambda x: ~x & 0xFF}
for k in range(1, 8):
    ATOMS[f"rol{k}"] = (lambda k: lambda x: rol(x, k))(k)
    ATOMS[f"ror{k}"] = (lambda k: lambda x: ror(x, k))(k)
    ATOMS[f"shl{k}"] = (lambda k: lambda x: (x << k) & 0xFF)(k)
    ATOMS[f"shr{k}"] = (lambda k: lambda x: x >> k)(k)
ATOMS["z0"] = lambda x: 0; ATOMS["z1"] = lambda x: 0xFF
LUT = {n: [f(x) for x in range(256)] for n, f in ATOMS.items()}
NAMES = list(LUT)
import itertools
def bit(v, k): return (v >> k) & 1

def parse_bit(prompt):
    exs = re.findall(r"([01]{8})\s*->\s*([01]{8})", prompt)
    mq = re.search(r"for:\s*([01]{8})", prompt)
    return [(int(a, 2), int(b, 2)) for a, b in exs], (int(mq.group(1), 2) if mq else None)

bits = []
for r in ROWS:
    e, q = parse_bit(r["prompt"])
    if len(e) >= 5 and q is not None and re.fullmatch(r"[01]{8}", str(r["answer"]).strip()):
        bits.append((e, q, int(str(r["answer"]).strip(), 2)))
print(f"bit puzzles: {len(bits)} (sampling 150 for speed)")

def determined(e, q, gold):
    for tn in itertools.combinations(NAMES, 3):
        L = [LUT[n] for n in tn]; c = {}; ok = True
        for inp, out in e:
            for k in range(8):
                key = (bit(L[0][inp], k) << 2) | (bit(L[1][inp], k) << 1) | bit(L[2][inp], k)
                v = bit(out, k)
                if c.get(key, v) != v: ok = False; break
                c[key] = v
            if not ok: break
        if not ok: continue
        o = 0; cov = True
        for k in range(8):
            key = (bit(L[0][q], k) << 2) | (bit(L[1][q], k) << 1) | bit(L[2][q], k)
            if key not in c: cov = False; break
            o |= c[key] << k
        if cov and o == gold: return True
    return False

samp = bits[:150]
det = sum(determined(e, q, g) for e, q, g in samp)
print(f"example-determined via full truth-table: {det}/{len(samp)} = {100*det/len(samp):.1f}%")
print("(offline determinacy ~99%. The wall is TRANSFER, not information: see write-up sec. 6-7.)")

bit puzzles: 1602 (sampling 150 for speed)
example-determined via full truth-table: 149/150 = 99.3%
(offline determinacy ~99%. The wall is TRANSFER, not information: see write-up sec. 6-7.)


## 3. gravity / unit_conversion — already 100% **under the 1% metric**
The "rounding ambiguity" reported elsewhere never crosses the 1% tolerance, so the
mean-coefficient estimator is already perfect under the *actual* metric -> no headroom.

In [6]:
def probe_linear(rows, kind):
    n = ok = 0
    for r in rows:
        p = r["prompt"]
        if kind == "gravity":
            if "0.5*g*t" not in p and "falling distance" not in p: continue
            ex = re.findall(r"t\s*=\s*([\d.]+).{0,20}?distance\s*=\s*([\d.]+)", p)
            mq = re.search(r"for\s*t\s*=\s*([\d.]+)", p)
            if not ex or not mq: continue
            ests = [2*float(d)/float(t)**2 for t, d in ex if float(t) > 0]
            if not ests: continue
            g = sum(ests)/len(ests); pred = 0.5*g*float(mq.group(1))**2
        else:
            ex = re.findall(r"([\d.]+)\s*m\s*becomes\s*([\d.]+)", p)
            mq = re.search(r"following measurement:\s*([\d.]+)", p)
            if not ex or not mq: continue
            ests = [float(y)/float(x) for x, y in ex if float(x) > 0]
            if not ests: continue
            k = sum(ests)/len(ests); pred = k*float(mq.group(1))
        n += 1; ok += verify(r["answer"], f"{pred:.2f}")
    return ok, n

for kind in ("gravity", "units"):
    ok, n = probe_linear(ROWS, kind)
    print(f"{kind:8}: {ok}/{n} correct under 1% metric = {100*ok/max(n,1):.1f}%")

gravity : 1597/1597 correct under 1% metric = 100.0%
units   : 1594/1594 correct under 1% metric = 100.0%


## 4. The pipeline (how the 0.85 adapter is produced)
```
problems -> per-family solvers/reasoners -> CoT kept iff verify(gold, boxed)
         -> LoRA rank-32 SFT (train mlp+attn+unembed)
         -> convert to Kaggle format (un-fuse MoE experts; merge Mamba gate+x_proj
            -> in_proj via QR+SVD; 12010 tensors) -> submission.zip
```
Train on the harness format verbatim (same SUFFIX + chat template, enable_thinking).
Use deductive, *observable* CoT only — declared conclusions do not transfer.

## 5. Experiment log (every attempt, including the failures)
| # | Experiment | Result | Lesson |
|---|---|---|---|
| 1 | rebuild all families (LoRA SFT) | **v2 = 0.84 LB** | controlled pipeline |
| 2 | + cipher synth / bit all / eqnum-MAP | **v3 = 0.85 LB** | matched baseline |
| 3 | cipher eval @3500 vs @7680 | 60% -> **95%** | evaluate at the harness cap |
| 4 | gravity/units under 1% metric | **100%** | no headroom (reproduced above) |
| 5 | eq_numeric blind-MAP ceiling | **78.6%** | ~at ceiling |
| 6 | eq_symbol example-consistent (honest) | **~8%** | offline-solvable != learnable |
| 7-9 | eq_symbol round-1 (string-rule / MAP / operator-independent) | 7.2% / 0% | (wrong rule model) |
| 17-20 | eq_symbol round-2 RE: global-map / digit-sub / under-det. / **prior** | 0/26 · 0/40 · 4 unique · **10.5% held-out** | under-determined; a learned prior recovers ~10% (reproduced live in 1b) |
| 10 | bit full truth-table (offline) | **99%** | determined in principle |
| 11-14 | bit transfer: guess/declared/staged/rung | 12.5 / 25 / 27.5 / cratered | **bit = capability wall** |
| 15 | recipe 1 epoch vs 3 | under-trains bit | -1.7pt is distribution |
| 16 | heldout -> LB calibration | constant **-1.7pt** | LB ~= projection - 1.7 |

**Takeaway.** 0.85 is, for this frozen base, close to the supervised ceiling: cipher /
deterministic ~100%, eq_numeric ~78%, eq_symbol under-determined (a learned prior reaches
~10% held-out, then cross-category forgetting gates the family near 0.86), bit ~84%
(transfer-bounded). The reusable contribution is this map and the reasons behind it.